In [15]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# ==========================================
# 1. ЗАГРУЗКА ЛОКАЛЬНЫХ ФАЙЛОВ И СКАНИРОВАНИЕ
# ==========================================
print("📦 Шаг 1. Загрузка локальных файлов...")
if not os.path.exists("train.csv") or not os.path.exists("test.csv"):
    raise FileNotFoundError("❌ Файлы 'train.csv' и/или 'test.csv' не найдены в Colab!")

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

def find_column(df, keywords):
    for col in df.columns:
        if any(kw.lower() in col.lower().replace("_", " ") for kw in keywords):
            return col
    return None

wifi_col = find_column(train_df, ["wifi"])
food_col = find_column(train_df, ["food", "drink"])
seat_col = find_column(train_df, ["seat", "comfort"])
ent_col = find_column(train_df, ["entertainment"])
travel_col = find_column(train_df, ["type", "travel"])
boarding_col = find_column(train_df, ["online", "boarding"])
dep_delay_col = find_column(train_df, ["departure", "delay"])
arr_delay_col = find_column(train_df, ["arrival", "delay"])
target_col = find_column(train_df, ["satisfact"])

if arr_delay_col:
    train_df[arr_delay_col] = train_df[arr_delay_col].fillna(0)
    test_df[arr_delay_col] = test_df[arr_delay_col].fillna(0)

# ==========================================
# 2. ВЕЛИКИЙ НАУЧНЫЙ КОНВЕЙЕР (7 МОДУЛЕЙ)
# ==========================================
def apply_all_scientific_goodies(df):
    # 1. Энтропия Шеннона
    def row_entropy(row):
        counts = np.bincount(row, minlength=6)[1:]
        probs = counts / np.sum(counts)
        probs = probs[probs > 0]
        return -np.sum(probs * np.log2(probs))

    service_cols = [wifi_col, food_col, seat_col, ent_col]
    df["Passenger_Opinion_Entropy"] = np.array([row_entropy(row) for row in df[service_cols].values])

    # 2. Энергия Гиббса
    H = 20 - df[service_cols].sum(axis=1).values
    dep_del = df[dep_delay_col].values if dep_delay_col else np.zeros(len(df))
    arr_del = df[arr_delay_col].values if arr_delay_col else np.zeros(len(df))
    total_delay = dep_del + arr_del
    S = np.log1p(total_delay)
    travel_series = df[travel_col].astype(str).str.lower()
    T = np.where(travel_series.str.contains("personal"), 2.5, 1.0)
    df["Gibbs_Free_Energy"] = H - (T * S)

    # 3. Центральность графа
    wifi = df[wifi_col].values
    entertainment = df[ent_col].values
    seat = df[seat_col].values
    df["Graph_Wifi_Centrality"] = 1.0 / (np.abs(wifi - entertainment) + np.abs(wifi - seat) + 1.0)

    # 4. Генетический алгоритм
    gene_a = df[wifi_col].values.astype(np.uint8)
    gene_b = df[ent_col].values.astype(np.uint8)
    crossover_gene = (gene_a & 0b1100) | (gene_b & 0b0011)
    np.random.seed(42)
    mutation_mask = np.random.choice([0b0000, 0b0001], size=len(df), p=[0.9, 0.1])
    df["Genetic_Evolved_Feature"] = crossover_gene ^ mutation_mask

    # 5. Аттрактор Лоренца (Теория Хаоса)
    sigma, rho, beta = 10.0, 28.0, 8.0 / 3.0
    x_chaos = (wifi - entertainment).astype(float)
    y_chaos = S.astype(float)
    z_chaos = np.where(travel_series.str.contains("personal"), 15.0, 45.0)
    dx = sigma * (y_chaos - x_chaos)
    dy = x_chaos * (rho - z_chaos) - y_chaos
    dz = x_chaos * y_chaos - beta * z_chaos
    df["Chaos_Attractor_Distance"] = np.round(np.sqrt(dx**2 + dy**2 + dz**2), 2)

    # 6. Теория Струн (Суперсимметричный резонанс)
    d_wifi = (5 - wifi).astype(float)
    d_food = (5 - df[food_col].values).astype(float)
    d_seat = (5 - seat).astype(float)
    d_ent = (5 - entertainment).astype(float)
    susy_invariant = (d_wifi * d_food) + (d_seat * d_ent) - (d_wifi * d_seat)
    df["String_SUSY_Resonance_Index"] = np.round(np.tanh(susy_invariant / 10.0), 4)

    # 7. Общая Теория Относительности (ОТО)
    gravity_mass = total_delay.astype(float)
    time_dilation_factor = 1.0 / np.sqrt(1.0 + (gravity_mass / 30.0))
    df["Relativistic_Time_Dilation"] = np.round(time_dilation_factor, 4)

    return df

print("🧠 Шаг 2. Расчет всех 7 научных признаков...")
train_df = apply_all_scientific_goodies(train_df)
test_df = apply_all_scientific_goodies(test_df)

le_travel = LabelEncoder()
train_df["encoded_travel"] = le_travel.fit_transform(train_df[travel_col].astype(str))
test_df["encoded_travel"] = le_travel.transform(test_df[travel_col].astype(str))

le_target = LabelEncoder()
y = le_target.fit_transform(train_df[target_col].astype(str))

# Базовое ядро фичей автора
base_core = ["encoded_travel", wifi_col, boarding_col]

# Полный список ВСЕХ 7 научных фичей для детальной изоляции
scientific_features = {
    "1. Энтропия Шеннона (Opinion Entropy)": "Passenger_Opinion_Entropy",
    "2. Энергия Гиббса (Thermodynamics)": "Gibbs_Free_Energy",
    "3. Центральность графа (Topology)": "Graph_Wifi_Centrality",
    "4. Генетическая мутация (Evolution)": "Genetic_Evolved_Feature",
    "5. Аттрактор Лоренца (Chaos Theory)": "Chaos_Attractor_Distance",
    "6. Теория Струн (SUSY Resonance)": "String_SUSY_Resonance_Index",
    "7. Эйнштейновская ОТО (Time Dilation)": "Relativistic_Time_Dilation"
}

# ==========================================
# 3. ЧЕСТНЫЙ КРОСС-ВАЛИДАЦИОННЫЙ СРАВНИТЕЛЬНЫЙ АНАЛИЗ
# ==========================================
print("\n🔥 Шаг 3. Запуск изолированного 5-Fold тестирования для всех 7 метрик...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
report_data = []

# --- Расчет Базового контроля ---
oof_base = np.zeros(len(train_df))
for train_idx, val_idx in kf.split(train_df, y):
    model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric="logloss")
    model.fit(train_df[base_core].iloc[train_idx], y[train_idx])
    oof_base[val_idx] = model.predict_proba(train_df[base_core].iloc[val_idx])[:, 1]

base_auc = roc_auc_score(y, oof_base)
report_data.append({"Метрика": "Чистый Базис Автора (Контроль)", "OOF ROC-AUC": base_auc, "Эффект": "0.00000"})
print(f"✅ Базовый контроль посчитан: {base_auc:.5f}")

# --- Расчет для каждой фичи поочередно ---
for name, col_name in scientific_features.items():
    current_features = base_core + [col_name]
    oof_current = np.zeros(len(train_df))

    for train_idx, val_idx in kf.split(train_df, y):
        model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric="logloss")
        model.fit(train_df[current_features].iloc[train_idx], y[train_idx])
        oof_current[val_idx] = model.predict_proba(train_df[current_features].iloc[val_idx])[:, 1]

    auc_score = roc_auc_score(y, oof_current)
    diff = auc_score - base_auc
    report_data.append({
        "Метрика": name,
        "OOF ROC-AUC": auc_score,
        "Эффект": f"+{diff:.5f}" if diff >= 0 else f"{diff:.5f}"
    })
    print(f"✅ Фича '{col_name}' обработана: {auc_score:.5f}")

# ==========================================
# 4. ИТОГОВЫЙ НАУЧНЫЙ ОТЧЕТ
# ==========================================
df_report = pd.DataFrame(report_data)
print("\n📊 " + "=" * 15 + " ПОЛНЫЙ НАУЧНЫЙ ОТЧЕТ КАЧЕСТВА (7 ФИЧЕЙ НА 5-FOLD CV) " + "=" * 15)
print(df_report.to_string(index=False))
print("=" * 86 + "\n")


📦 Шаг 1. Загрузка локальных файлов...
🧠 Шаг 2. Расчет всех 7 научных признаков...

🔥 Шаг 3. Запуск изолированного 5-Fold тестирования для всех 7 метрик...
✅ Базовый контроль посчитан: 0.80560
✅ Фича 'Passenger_Opinion_Entropy' обработана: 0.85759
✅ Фича 'Gibbs_Free_Energy' обработана: 0.92682
✅ Фича 'Graph_Wifi_Centrality' обработана: 0.87756
✅ Фича 'Genetic_Evolved_Feature' обработана: 0.84916
✅ Фича 'Chaos_Attractor_Distance' обработана: 0.92997
✅ Фича 'String_SUSY_Resonance_Index' обработана: 0.85653
✅ Фича 'Relativistic_Time_Dilation' обработана: 0.90827

📊 =============== ПОЛНЫЙ НАУЧНЫЙ ОТЧЕТ КАЧЕСТВА (7 ФИЧЕЙ НА 5-FOLD CV) ===============
                              Метрика  OOF ROC-AUC   Эффект
       Чистый Базис Автора (Контроль)     0.805600  0.00000
1. Энтропия Шеннона (Opinion Entropy)     0.857585 +0.05199
   2. Энергия Гиббса (Thermodynamics)     0.926821 +0.12122
    3. Центральность графа (Topology)     0.877559 +0.07196
  4. Генетическая мутация (Evolution)     0.849

In [13]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [14]:
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# ==========================================
# 1. ЗАГРУЗКА ЛОКАЛЬНЫХ ФАЙЛОВ
# ==========================================
print("📦 Шаг 1. Загрузка локальных файлов...")
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


def find_column(df, keywords):
    for col in df.columns:
        if any(kw.lower() in col.lower().replace("_", " ") for kw in keywords):
            return col
    return None


wifi_col = find_column(train_df, ["wifi"])
food_col = find_column(train_df, ["food", "drink"])
seat_col = find_column(train_df, ["seat", "comfort"])
ent_col = find_column(train_df, ["entertainment"])
travel_col = find_column(train_df, ["type", "travel"])
boarding_col = find_column(train_df, ["online", "boarding"])
dep_delay_col = find_column(train_df, ["departure", "delay"])
arr_delay_col = find_column(train_df, ["arrival", "delay"])
target_col = find_column(train_df, ["satisfact"])

if arr_delay_col:
    train_df[arr_delay_col] = train_df[arr_delay_col].fillna(0)
    test_df[arr_delay_col] = test_df[arr_delay_col].fillna(0)


# ==========================================
# 2. ВЕЛИКИЙ НАУЧНЫЙ КОНВЕЙЕР (7 МОДУЛЕЙ)
# ==========================================
def apply_all_scientific_goodies(df):
    # 1. ТЕОРИЯ ИНФОРМАЦИИ (ЭНТРОПИЯ ШЕННОНА)
    def row_entropy(row):
        counts = np.bincount(row, minlength=6)[1:]
        probs = counts / np.sum(counts)
        probs = probs[probs > 0]
        return -np.sum(probs * np.log2(probs))

    service_cols = [wifi_col, food_col, seat_col, ent_col]
    df["Passenger_Opinion_Entropy"] = np.array([
        row_entropy(row) for row in df[service_cols].values
    ])

    # 2. ТЕРМОДИНАМИКА ПОЛЕТА (ЭНЕРГИЯ ГИББСА)
    H = 20 - df[service_cols].sum(axis=1).values
    dep_del = df[dep_delay_col].values if dep_delay_col else np.zeros(len(df))
    arr_del = df[arr_delay_col].values if arr_delay_col else np.zeros(len(df))
    total_delay = dep_del + arr_del
    S = np.log1p(total_delay)
    travel_series = df[travel_col].astype(str).str.lower()
    T = np.where(travel_series.str.contains("personal"), 2.5, 1.0)
    df["Gibbs_Free_Energy"] = H - (T * S)

    # 3. ТЕОРИЯ ГРАФОВ (ТОПОЛОГИЧЕСКАЯ ЦЕНТРАЛЬНОСТЬ)
    wifi = df[wifi_col].values
    entertainment = df[ent_col].values
    seat = df[seat_col].values
    df["Graph_Wifi_Centrality"] = 1.0 / (
        np.abs(wifi - entertainment) + np.abs(wifi - seat) + 1.0
    )

    # 4. ГЕНЕТИЧЕСКИЙ АЛГОРИТМ (ЭВОЛЮЦИЯ БИТОВ)
    gene_a = df[wifi_col].values.astype(np.uint8)
    gene_b = df[ent_col].values.astype(np.uint8)
    crossover_gene = (gene_a & 0b1100) | (gene_b & 0b0011)
    np.random.seed(42)
    mutation_mask = np.random.choice(
        [0b0000, 0b0001], size=len(df), p=[0.9, 0.1]
    )
    df["Genetic_Evolved_Feature"] = crossover_gene ^ mutation_mask

    # 5. ТЕОРИЯ ХАОСА (АТТРАКТОР ЛОРЕНЦА)
    sigma, rho, beta = 10.0, 28.0, 8.0 / 3.0
    x_chaos = (wifi - entertainment).astype(float)
    y_chaos = S.astype(float)
    z_chaos = np.where(travel_series.str.contains("personal"), 15.0, 45.0)
    dx = sigma * (y_chaos - x_chaos)
    dy = x_chaos * (rho - z_chaos) - y_chaos
    dz = x_chaos * y_chaos - beta * z_chaos
    df["Chaos_Attractor_Distance"] = np.round(np.sqrt(dx**2 + dy**2 + dz**2), 2)

    # 6. ТЕОРИЯ СТРУН (СУПЕРСИММЕТРИЧНЫЙ РЕЗОНАНС)
    d_wifi = (5 - wifi).astype(float)
    d_food = (5 - df[food_col].values).astype(float)
    d_seat = (5 - seat).astype(float)
    d_ent = (5 - entertainment).astype(float)
    susy_invariant = (d_wifi * d_food) + (d_seat * d_ent) - (d_wifi * d_seat)
    df["String_SUSY_Resonance_Index"] = np.round(np.tanh(susy_invariant / 10.0), 4)

    # 7. ОБЩАЯ ТЕОРИЯ ОТНОСИТЕЛЬНОСТИ (ОТО)
    gravity_mass = total_delay.astype(float)
    time_dilation_factor = 1.0 / np.sqrt(1.0 + (gravity_mass / 30.0))
    df["Relativistic_Time_Dilation"] = np.round(time_dilation_factor, 4)

    return df


print("🧠 Шаг 2. Расчет 7 научных модулей...")
train_df = apply_all_scientific_goodies(train_df)
test_df = apply_all_scientific_goodies(test_df)

le_travel = LabelEncoder()
train_df["encoded_travel"] = le_travel.fit_transform(
    train_df[travel_col].astype(str)
)
test_df["encoded_travel"] = le_travel.transform(test_df[travel_col].astype(str))

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[target_col].astype(str))

# Финальный пул признаков для ансамбля
mega_features = [
    "encoded_travel",
    wifi_col,
    boarding_col,
    "Passenger_Opinion_Entropy",
    "Gibbs_Free_Energy",
    "Graph_Wifi_Centrality",
    "Genetic_Evolved_Feature",
    "Chaos_Attractor_Distance",
    "String_SUSY_Resonance_Index",
    "Relativistic_Time_Dilation",
]

X = train_df[mega_features]
y = y_train

# ==========================================
# 3. КРОСС-ВАЛИДАЦИОННЫЙ ТРОЙНОЙ АНСАМБЛЬ
# ==========================================
print(
    f"\n🛡️ Шаг 3. Запуск 5-Fold Кросс-Валидации Тройного Ансамбля (CatBoost + XGBoost + LightGBM)..."
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Массивы для хранения OOF-предсказаний каждой модели
oof_cat = np.zeros(len(train_df))
oof_xgb = np.zeros(len(train_df))
oof_lgb = np.zeros(len(train_df))
oof_ensemble = np.zeros(len(train_df))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f" 📂 Обработка Фолда {fold + 1}/5...")
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]

    # Инициализация моделей фолда
    model_cat = CatBoostClassifier(
        iterations=150, depth=6, learning_rate=0.1, random_seed=42, verbose=0
    )
    model_xgb = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric="logloss"
    )
    model_lgb = LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=-1
    )

    # Обучение триумвирата
    model_cat.fit(X_tr, y_tr)
    model_xgb.fit(X_tr, y_tr)
    model_lgb.fit(X_tr, y_tr)

    # Валидационные предсказания (Out-of-Fold)
    p_cat = model_cat.predict_proba(X_va)[:, 1]
    p_xgb = model_xgb.predict_proba(X_va)[:, 1]
    p_lgb = model_lgb.predict_proba(X_va)[:, 1]

    oof_cat[val_idx] = p_cat
    oof_xgb[val_idx] = p_xgb
    oof_lgb[val_idx] = p_lgb

    # Взвешенное смешивание на текущем фолде (40% Cat + 35% XGB + 25% LGB)
    oof_ensemble[val_idx] = 0.40 * p_cat + 0.35 * p_xgb + 0.25 * p_lgb

    fold_auc = roc_auc_score(y_va, oof_ensemble[val_idx])
    print(f"      ↳ Смешанный ROC-AUC фолда {fold + 1} = {fold_auc:.5f}")

# ==========================================
# 4. ИТОГОВЫЙ НАУЧНЫЙ ОТЧЕТ АНСАМБЛЯ
# ==========================================
print("\n📊 " + "=" * 10 + " ГЛОБАЛЬНЫЙ ОТЧЕТ СИНЕРГЕТИЧЕСКОГО СТЕКА " + "=" * 10)
print(f"🔹 Глобальный OOF ROC-AUC (только CatBoost):  {roc_auc_score(y, oof_cat):.5f}")
print(f"🔹 Глобальный OOF ROC-AUC (только XGBoost):   {roc_auc_score(y, oof_xgb):.5f}")
print(f"🔹 Глобальный OOF ROC-AUC (только LightGBM):  {roc_auc_score(y, oof_lgb):.5f}")
print("-" * 62)
print(f"🔸 ИТОГОВЫЙ ТРОЙНОЙ АНСАМБЛЬ OOF ROC-AUC:      {roc_auc_score(y, oof_ensemble):.5f}")
print("=" * 62 + "\n")


📦 Шаг 1. Загрузка локальных файлов...
🧠 Шаг 2. Расчет 7 научных модулей...

🛡️ Шаг 3. Запуск 5-Fold Кросс-Валидации Тройного Ансамбля (CatBoost + XGBoost + LightGBM)...
 📂 Обработка Фолда 1/5...
      ↳ Смешанный ROC-AUC фолда 1 = 0.93745
 📂 Обработка Фолда 2/5...
      ↳ Смешанный ROC-AUC фолда 2 = 0.93749
 📂 Обработка Фолда 3/5...
      ↳ Смешанный ROC-AUC фолда 3 = 0.93805
 📂 Обработка Фолда 4/5...
      ↳ Смешанный ROC-AUC фолда 4 = 0.93788
 📂 Обработка Фолда 5/5...
      ↳ Смешанный ROC-AUC фолда 5 = 0.93883

📊 ========== ГЛОБАЛЬНЫЙ ОТЧЕТ СИНЕРГЕТИЧЕСКОГО СТЕКА ==========
🔹 Глобальный OOF ROC-AUC (только CatBoost):  0.93793
🔹 Глобальный OOF ROC-AUC (только XGBoost):   0.93743
🔹 Глобальный OOF ROC-AUC (только LightGBM):  0.93746
--------------------------------------------------------------
🔸 ИТОГОВЫЙ ТРОЙНОЙ АНСАМБЛЬ OOF ROC-AUC:      0.93792

